In [15]:
import redis
import time
import json

In [12]:
# 连接 Redis
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

In [25]:
## 基于List的任务队列

# 生产者推送任务
def push_task(task):
    r.rpush("task_queue__", task)
    print(f"任务 {task} 已加入队列")

# 测试
push_task(json.dumps({"key1":"task_1"}))
push_task(json.dumps({"key2":"task_2"}))

# 消费者处理任务
def process_tasks():
    while True:
        task = r.blpop("task_queue__", timeout=10)  # 阻塞式等待任务
        if task:
            _, task_value = task
            print(f"处理任务: {task_value}")
        else:
            print("无任务，等待中...")

# 启动消费者
process_tasks()



任务 {"key1": "task_1"} 已加入队列
任务 {"key2": "task_2"} 已加入队列
处理任务: {"key1": "task_1"}
处理任务: {"key2": "task_2"}
无任务，等待中...


KeyboardInterrupt: 

# 基于Pub/Sub（发布订阅）

In [ ]:
## 基于Pub/Sub（发布订阅）

def publish_message(channel, message):
    r.publish(channel, message)
    print(f"消息发送到 {channel}: {message}")

# 测试
publish_message("news", "Breaking news: Redis is amazing!")



消息发送到 news: Breaking news: Redis is amazing!


In [22]:
# pubsub = r.pubsub()
# pubsub.subscribe("news")  # 订阅 news 频道
# pubsub.subscribe("class")  # 订阅 news 频道


# print("等待消息中...")
# for message in pubsub.listen():
#     if message['type'] == 'message':
#         print(f"收到消息: {message['data']}")

# 基于Streams（可靠的消息队列）

In [40]:
#生产者推送任务
def push_task(task):
    task_id = r.xadd("task_stream", {"task": task})
    print(f"任务 {task} 已添加到 Stream，ID: {task_id}")

# 测试
push_task("stream_task_1")
push_task("stream_task_2")
push_task("stream_task_3")
push_task("stream_task_4")
push_task("stream_task_5")
push_task("stream_task_6")
push_task("stream_task_7")
push_task("stream_task_8")
push_task("stream_task_9")
push_task("stream_task_10")

任务 stream_task_1 已添加到 Stream，ID: 1742462927610-0
任务 stream_task_2 已添加到 Stream，ID: 1742462927611-0
任务 stream_task_3 已添加到 Stream，ID: 1742462927611-1
任务 stream_task_4 已添加到 Stream，ID: 1742462927611-2
任务 stream_task_5 已添加到 Stream，ID: 1742462927612-0
任务 stream_task_6 已添加到 Stream，ID: 1742462927612-1
任务 stream_task_7 已添加到 Stream，ID: 1742462927612-2
任务 stream_task_8 已添加到 Stream，ID: 1742462927612-3
任务 stream_task_9 已添加到 Stream，ID: 1742462927612-4
任务 stream_task_10 已添加到 Stream，ID: 1742462927613-0


In [29]:
# #会重复读任务
# while True:
#     tasks = r.xread({"task_stream": "0"}, count=10, block=5000)  # 读取所有任务
#     for stream, messages in tasks:
#         for message_id, message in messages:
#             print(f"消费者1 处理任务: {message['task']}")


STREAM_NAME = "task_stream"
last_id = "0"  # 记录上次读取的位置

while True:
    tasks = r.xread({STREAM_NAME: last_id}, count=10, block=5000)

    for stream, messages in tasks:
        for message_id, message in messages:
            print(f"消费者1 处理任务: {message['task']}")
            last_id = message_id  # 关键！更新最后读取的位置，防止重复读取


消费者1 处理任务: stream_task_1
消费者1 处理任务: stream_task_2


KeyboardInterrupt: 

In [41]:
GROUP_NAME = "workers"
CONSUMER_NAME = "worker_1"
STREAM_NAME = "task_stream"
# 确保消费者组存在（第一次运行时需要创建）
try:
    r.xgroup_create(STREAM_NAME, GROUP_NAME, id='0', mkstream=True)
except redis.exceptions.ResponseError:
    pass  # 消费者组已存在
# 读取任务
def consume_tasks():
    while True:
        time.sleep(1)
        tasks = r.xreadgroup(GROUP_NAME, CONSUMER_NAME, {STREAM_NAME: ">"}, 
                             count=2, block=2000)
        if tasks:
            print(f"任务数：{len(tasks)}")
            for stream, messages in tasks:
                for message_id, message in messages:
                    print(f"消费任务: {message['task']}")
                    r.xack(STREAM_NAME, GROUP_NAME, message_id)  # 确认任务已处理
        else:
            print("等待任务中...")
# 启动消费者
consume_tasks()


任务数：1
消费任务: stream_task_1
消费任务: stream_task_2
任务数：1
消费任务: stream_task_3
消费任务: stream_task_4
任务数：1
消费任务: stream_task_6
消费任务: stream_task_7
任务数：1
消费任务: stream_task_9
消费任务: stream_task_10
等待任务中...
等待任务中...
等待任务中...


KeyboardInterrupt: 